# Train the TVD detector on Google Colab

Runs the Phase 1 fine-tune (yolov8n on UA-DETRAC-DATASET-10K) on a Colab GPU.

**Before running:** in Colab, open the key/secrets panel (left sidebar) and add
`ROBOFLOW_API_KEY` with your key from https://app.roboflow.com/settings/api.
The key stays in Colab secrets — it is never written to the notebook or repo.

**After training:** download `weights/detector.pt` and `docs/RESULTS.md`
(cell at the bottom does this) and copy them into your local repo checkout.

In [ ]:
# 1. Clone the repo (branch with the detection pipeline) and install deps
!git clone -b phase-1-detection https://github.com/Rakib-codee/Traffic-Violation-Detection.git tvd
%cd tvd
!pip install -q -e ".[dev,data]"

In [ ]:
# 2. Read the Roboflow API key from Colab secrets
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("ROBOFLOW_API_KEY loaded from secrets")

In [ ]:
# 3. Download the vehicle dataset (UA-DETRAC-DATASET-10K v2, YOLOv8 format)
!python scripts/download_dataset.py \
    --workspace rjacaac1 --project ua-detrac-dataset-10k --version 2 \
    --format yolov8 --location data/vehicles

In [ ]:
# 4. Sanity-check the dataset (9,316 train / 500 val images expected)
!python -c "from tvd.detect.data import validate_dataset; \
i = validate_dataset('data/vehicles/data.yaml'); \
print(i.n_train, 'train /', i.n_val, 'val /', i.names)"

In [ ]:
# 5. Train (20 epochs, CUDA). Seed and config are logged to results/detection/detector/.
!python scripts/train_detector.py --train-config configs/train.yaml --device cuda

In [ ]:
# 6. Evaluate: regenerates docs/RESULTS.md with the mAP table
!python scripts/eval_detector.py --weights weights/detector.pt \
    --train-config configs/train.yaml --device cuda

In [ ]:
# 7. Download the artifacts back to your machine
from google.colab import files

files.download("weights/detector.pt")
files.download("docs/RESULTS.md")